In [1]:
# Task 3.1
import sqlite3

conn = sqlite3.connect('minesweeper.db')
cursor = conn.cursor()

with conn:
    cursor.executescript('''
        CREATE TABLE IF NOT EXISTS Board(
            xCoordinate INT NOT NULL,
            yCoordinate INT NOT NULL,
            hasMine INT NOT NULL,
            checked INT NOT NULL,

            PRIMARY KEY (xCoordinate, yCoordinate)
        );
    ''')
    
    conn.commit()

In [2]:
# Task 3.2
from socket import socket
import sqlite3
from random import sample

s = socket()

s.bind(('', 6789))

s.listen()

conn = sqlite3.connect('minesweeper.db')
cursor = conn.cursor()

with conn:
    cursor.execute('SELECT xCoordinate, yCoordinate FROM Board')
    board = cursor.fetchall()

    for tile in board:
        cursor.execute('DELETE FROM Board WHERE xCoordinate = ? AND yCoordinate = ?', tile)

    mines = sample([i for i in range(1, 17)], 10)

    tile_no = 1
    for x in range(1, 5):
        for y in range(1, 5):
            if tile_no in mines:
                is_mine = 1
            else:
                is_mine = 0
            
            cursor.execute('INSERT INTO Board VALUES (?, ?, ?, ?)', (x, y, is_mine, 0))

            tile_no += 1

    conn.commit()
    
    s.listen()
    c, addr = s.accept()

    checked = 0
    while checked < 6:
        coords = c.recv(4096).decode()
        x, y = [int(i) for i in coords.split(',')]

        cursor.execute('UPDATE Board SET checked = 1 WHERE xCoordinate = ? AND yCoordinate = ?', (x, y))
        conn.commit()
        
        cursor.execute('SELECT hasMine FROM Board WHERE xCoordinate = ? AND yCoordinate = ?', (x, y))
        is_mine = cursor.fetchone()[0]

        if is_mine == 1:
            c.send('MINE'.encode())
            print('MINE HIT!')
            break
        else:
            cursor.execute('SELECT Count(checked) FROM Board')
            checked = cursor.fetchone()[0]

            surrounding_mines = 0
            for i in range(-1, 2):
                for j in range(-1, 2):
                    if i == j == 0: continue
                    
                    x_offset = x + i
                    y_offset = y + j

                    cursor.execute('SELECT hasMine FROM Board WHERE xCoordinate = ? AND yCoordinate = ?', (x_offset, y_offset))
                    is_mine = cursor.fetchone()[0]

                    if is_mine == 1:
                        surrounding_mines += 1

            c.send(f'{surrounding_mines},{checked}'.encode())

OSError: [Errno 98] Address already in use

In [3]:
import random; help(random.sample)

Help on method sample in module random:

sample(population, k, *, counts=None) method of random.Random instance
    Chooses k unique random elements from a population sequence.

    Returns a new list containing elements from the population while
    leaving the original population unchanged.  The resulting list is
    in selection order so that all sub-slices will also be valid random
    samples.  This allows raffle winners (the sample) to be partitioned
    into grand prize and second place winners (the subslices).

    Members of the population need not be hashable or unique.  If the
    population contains repeats, then each occurrence is a possible
    selection in the sample.

    Repeated elements can be specified one at a time or with the optional
    counts parameter.  For example:

        sample(['red', 'blue'], counts=[4, 2], k=5)

    is equivalent to:

        sample(['red', 'red', 'red', 'red', 'blue', 'blue'], k=5)

    To choose a sample from a range of integers, use 